# 概览

```text
labs/MuJoCo/
├── main.py                              # IK 求解 + MuJoCo 可视化（最小示例）
├── ik.py                                # 纯 IK 求解与 3D 连杆姿态绘图（matplotlib）
├── push.py                              # 完整推箱实验：轨迹规划 + 接触力采集/可视化
├── ur5e_joint_linear_interpolation.py   # 关节空间线性插值轨迹控制（无接触力）
└── model/                               # UR5e 机器人模型资源
    ├── ur5e.urdf                        # xacro 自动生成的 URDF，供 ikpy 做运动学求解
    └── universal_robots_ur5e/           # MuJoCo 官方 UR5e MJCF 模型包
        ├── scene.xml                    # 场景入口：地面/天空/光照 + 坐标轴 + 红色方块
        ├── ur5e.xml                     # 机器人本体：关节、执行器、力传感器、几何体
        └── assets/                      # 各连杆的 .obj 网格（20 个，见下）
```

重点：

- labs\MuJoCo\model\universal_robots_ur5e\ur5e.xml：UR5e 机器人模型 MJCF 文件
- labs\MuJoCo\model\universal_robots_ur5e\scene.xml：UR5e 机器人模型场景文件

- labs\MuJoCo\model\ur5e.urdf：UR5e 机器人模型 URDF 文件

# URDF & MJCF

| 维度 | URDF（ur5e.urdf） | MJCF（ur5e.xml） |
|---|---|---|
| 来源/生态 | ROS 事实标准，由`xacro` 从`.xacro` 自动生成 | MuJoCo 原生格式，官方手工编辑维护 |
| 定位 | 纯运动学 + 几何 描述（关节树、link、visual/collision） | 面向物理仿真 的完整模型描述 |
| 物理引擎参数 | 无。求解器、积分器、接触参数都无处可写 | 有`<compiler>` 、`<option integrator="implicitfast">` 、`<default>` 模板、接触/摩擦参数 |
| 驱动与传感 | 无执行器概念（关节只有`<limit>` ），传感器靠`<gazebo>` 插件外挂 | 原生`<actuator>` （gain/bias/forcerange）、`<sensor>` ，本项目自定义了`force_sensor` 站点并输出接触力 |
| 场景内容 | 只有机器人本体，没有世界 | 可有资产纹理/材质、光照、地板、自由物体，并用`<include>` 组合出 scene.xml |
| 结构能力 | 严格树形，闭链需额外扩展；无`<include>` 分层 | 支持`include` 分层、`default` 类继承、equality/tendon/keyframe 等 |
| 本项目用途 | 喂给 ikpy 做逆运动学求解 | 喂给 MuJoCo 做物理仿真与渲染 |

差别在代码里的体现：

ik.py 用 URDF：

```python
my_chain = ikpy.chain.Chain.from_urdf_file(URDF_PATH)
```

main.py 用 MJCF：

```python
model = mujoco.MjModel.from_xml_path(MODEL_PATH)
```

- ikpy 只关心“关节角 ↔ 末端位姿”，URDF 的关节链信息就够了。
- MuJoCo 要算动力学、接触力、驱动力，必须要有质量/惯量/执行器/传感器，这些都只在 MJCF 里。

# Reference

[1] [MuJoCo 全流程实战教程：从零搭建一个仿真实验](https://blog.csdn.net/Vint_LU/article/details/147948556)